### Dim_Date

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Determine date range from Silver orders - pad a bit on both ends for safety
date_bounds = spark.sql("""
    SELECT 
        date_sub(min(order_purchase_timestamp), 30) as min_date,
        date_add(max(order_purchase_timestamp), 30) as max_date
    FROM ecommerce_dev.silver.orders
""").collect()[0]

min_date = date_bounds['min_date']
max_date = date_bounds['max_date']

print(f"Date range: {min_date} to {max_date}")

Date range: 2016-08-05 to 2018-11-16


### Generate calendar via sequence, then explode into one row per date

In [0]:
dim_date = spark.sql(f"""
    SELECT explode(sequence(to_date('{min_date}'), to_date('{max_date}'), interval 1 day)) as full_date
""")

dim_date = (dim_date
    .withColumn("date_key", date_format(col("full_date"), "yyyyMMdd").cast("int"))
    .withColumn("year", year(col("full_date")))
    .withColumn("quarter", quarter(col("full_date")))
    .withColumn("month", month(col("full_date")))
    .withColumn("month_name", date_format(col("full_date"), "MMMM"))
    .withColumn("day_of_month", dayofmonth(col("full_date")))
    .withColumn("day_of_week", dayofweek(col("full_date")))  # 1=Sunday
    .withColumn("day_name", date_format(col("full_date"), "EEEE"))
    .withColumn("week_of_year", weekofyear(col("full_date")))
    .withColumn("is_weekend", col("day_of_week").isin(1, 7))
    .select(
        "date_key", "full_date", "year", "quarter", "month", "month_name",
        "day_of_month", "day_of_week", "day_name", "week_of_year", "is_weekend"
    )
)

display(dim_date.limit(10))
print(f"Row count: {dim_date.count()}")

date_key,full_date,year,quarter,month,month_name,day_of_month,day_of_week,day_name,week_of_year,is_weekend
20160805,2016-08-05,2016,3,8,August,5,6,Friday,31,false
20160806,2016-08-06,2016,3,8,August,6,7,Saturday,31,true
20160807,2016-08-07,2016,3,8,August,7,1,Sunday,31,true
20160808,2016-08-08,2016,3,8,August,8,2,Monday,32,false
20160809,2016-08-09,2016,3,8,August,9,3,Tuesday,32,false
20160810,2016-08-10,2016,3,8,August,10,4,Wednesday,32,false
20160811,2016-08-11,2016,3,8,August,11,5,Thursday,32,false
20160812,2016-08-12,2016,3,8,August,12,6,Friday,32,false
20160813,2016-08-13,2016,3,8,August,13,7,Saturday,32,true
20160814,2016-08-14,2016,3,8,August,14,1,Sunday,32,true


Row count: 834


### Write to Gold - Dim_Date is fully regenerable, so overwrite is safe (no SCD needed)

In [0]:
dim_date.write.format("delta").mode("overwrite").saveAsTable("ecommerce_dev.gold.dim_date")

# Drop existing constraint if it exists
spark.sql("""
    ALTER TABLE ecommerce_dev.gold.dim_date 
    DROP CONSTRAINT IF EXISTS pk_dim_date
""")

# Set date_key to NOT NULL
spark.sql("""
    ALTER TABLE ecommerce_dev.gold.dim_date 
    ALTER COLUMN date_key SET NOT NULL
""")

# Add primary key constraint
spark.sql("""
    ALTER TABLE ecommerce_dev.gold.dim_date 
    ADD CONSTRAINT pk_dim_date PRIMARY KEY (date_key)
""")

spark.sql("""
    COMMENT ON TABLE ecommerce_dev.gold.dim_date IS 
    'Gold calendar dimension. Fully regenerable from order date range - no SCD tracking needed. Grain: one row per calendar date.'
""")

DataFrame[]